# 09 – Evaluation: Matriz de Confusión

**Proyecto:** Análisis y predicción del subempleo por insuficiencia de horas en el Perú – EPEN 2024  
**Target:** `target_subempleo_horas` (1 = subempleado por horas · 0 = no subempleado)  
**Objetivo:** Analizar la matriz de confusión del modelo ganador, interpretando FP y FN en el contexto del subempleo por horas.

> **Prerequisito:** Ejecuta primero `06_feature_selection/selected_variables.ipynb` y `07_modelling/01_baseline_model.ipynb`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import joblib
from pathlib import Path

# ── Rutas ──────────────────────────────────────────────────────────────────────
SEL_DIR   = Path('../data/selected')
MODEL_DIR = Path('../models')

TARGET      = 'target_subempleo_horas'
CLASS_NAMES = ['No subempleado\npor horas', 'Subempleado\npor horas']

# ── Carga de datos reales (sin fallback sintético) ─────────────────────────────
required = {
    'X_test':  SEL_DIR / 'X_test_selected.csv',
    'y_test':  SEL_DIR / 'y_test_selected.csv',
    'X_train': SEL_DIR / 'X_train_selected.csv',
    'y_train': SEL_DIR / 'y_train_selected.csv',
}
for name, path in required.items():
    if not path.exists():
        raise FileNotFoundError(
            f"Archivo requerido no encontrado: {path}\n"
            "Ejecuta primero: 06_feature_selection/selected_variables.ipynb"
        )

X_train = pd.read_csv(required['X_train'])
X_test  = pd.read_csv(required['X_test'])

def load_target(path, name):
    df = pd.read_csv(path)
    if TARGET in df.columns:
        return df[TARGET].reset_index(drop=True)
    if df.shape[1] == 1:
        return df.iloc[:, 0].reset_index(drop=True)
    raise ValueError(f"No se encontró '{TARGET}' en {name}")

y_train = load_target(required['y_train'], 'y_train')
y_test  = load_target(required['y_test'],  'y_test')

# ── Carga del modelo ganador ───────────────────────────────────────────────────
model_path = MODEL_DIR / 'logistic_regression_balanced.pkl'
if not model_path.exists():
    raise FileNotFoundError(
        f"Modelo no encontrado: {model_path}\n"
        "Ejecuta primero: 07_modelling/01_baseline_model.ipynb"
    )

model  = joblib.load(model_path)
y_pred = model.predict(X_test)
print(f'Modelo cargado : {model_path.name}')
print(f'Predicciones listas  (n={len(y_pred)}).')


## 1. Matriz de Confusión

In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print(f'Verdaderos Negativos  (TN): {tn:>5} – No subempleados correctamente clasificados')
print(f'Falsos Positivos      (FP): {fp:>5} – No subempleados clasificados como subempleados')
print(f'Falsos Negativos      (FN): {fn:>5} – Subempleados NO detectados ⚠️')
print(f'Verdaderos Positivos  (TP): {tp:>5} – Subempleados correctamente identificados')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de confusión – conteos
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=CLASS_NAMES)
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Matriz de Confusión – Conteos')

# Matriz de confusión – normalizada por fila
cm_norm = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', ax=axes[1],
            xticklabels=CLASS_NAMES,
            yticklabels=CLASS_NAMES)
axes[1].set_title('Matriz de Confusión – Normalizada')
axes[1].set_xlabel('Predicción')
axes[1].set_ylabel('Real')

plt.suptitle('Logistic Regression (class_weight="balanced") – Conjunto de prueba EPEN 2024',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.show()


## 2. Interpretación en contexto

| Error | Tipo | Impacto en política pública |
|---|---|---|
| **Falso Negativo (FN)** | Subempleado → No detectado | **Alto**: trabajadores con jornada insuficiente no reciben apoyo ni se visibilizan en estadísticas |
| **Falso Positivo (FP)** | No subempleado → Clasificado como subempleado | **Moderado**: sobreestimación del fenómeno; puede derivar en intervenciones no focalizadas |

> En este problema, **minimizar los Falsos Negativos** (maximizar Recall de clase 1) es la prioridad, dado que el subempleo por horas es un fenómeno que se busca identificar para política pública. La Regresión Logística balanceada alcanza un Recall de clase 1 ≈ 0.57, el más alto entre todos los modelos evaluados.


In [ ]:
# Métricas derivadas de la matriz de confusión
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
ppv = tp / (tp + fp) if (tp + fp) > 0 else 0  # Precision
npv = tn / (tn + fn) if (tn + fn) > 0 else 0  # Negative Predictive Value

print(f'Sensibilidad (Recall) : {sensitivity:.4f}')
print(f'Especificidad         : {specificity:.4f}')
print(f'Precisión (PPV)       : {ppv:.4f}')
print(f'NPV                   : {npv:.4f}')